In [1]:
import geemap
import ee

Map = geemap.Map()
Map

Map(center=[40, -100], controls=(WidgetControl(options=['position'], widget=HBox(children=(ToggleButton(value=…

In [2]:
def clip(img):
    return img.clip(roi)

In [3]:
roi = Map.draw_last_feature.geometry()

In [4]:
images = ee.ImageCollection("NASA/GIMMS/3GV0").select('ndvi')\
        .filterDate('2012', '2013').map(clip)

In [5]:
visParam = {
 'min': -1,
 'max': 1,
 'palette': ['000000', 'f5f5f5', '119701']
}
Map.addLayer(images, visParam, 'NDVI')

In [71]:
img = images.toBands()
ps = img.sample(region = roi, geometries = True)

In [9]:
Map.addLayer(ps, {}, 'ps')

In [83]:
order = 2
window_size = 11
def savitzky_golay(y):
    half_window = (window_size - 1) / 2
    deriv = 0
    order_range = ee.List.sequence(0, order)
    k_range = ee.List.sequence(-half_window, half_window)
    def fun1(k):
        return order_range.map(lambda o: ee.Number(k).pow(o))
    b = ee.Array(k_range.map(fun1))
    mPI = ee.Array(b.matrixPseudoInverse())
    impulse_response = (mPI.slice(**{ 'axis': 0, 'start': deriv, 'end': deriv + 1 })).project([1])
    y0 = y.get(0)
    firstvals = y.slice(1, half_window + 1).reverse().map(
        lambda e: ee.Number(e).subtract(y0).abs().multiply(-1).add(y0))
    yend = y.get(-1)
    lastvals = y.slice(-half_window - 1, -1).reverse().map(
        lambda e: ee.Number(e).subtract(yend).abs().add(yend))
    y_ext = firstvals.cat(y).cat(lastvals)
    runLength = ee.List.sequence(0, y_ext.length().subtract(window_size))
    smooth = runLength.map(
        lambda i:ee.Array(y_ext.slice(ee.Number(i), ee.Number(i).add(window_size))).multiply(impulse_response).reduce("sum", [0]).get([0])
    )
    return smooth

In [92]:
def fun(feature):
    List = bands.map(lambda i: feature.get(i))
    sg_ = savitzky_golay(List)
    return feature.set(ee.Dictionary.fromLists(bands_sg, sg_))

In [81]:
bands = img.bandNames()
List = bands.map(lambda i: ps.first().get(i))

In [91]:
bands_sg = bands.map(lambda i: ee.String(i).cat('_sg'))
bands_sg.getInfo()

['201201a_ndvi_sg',
 '201201b_ndvi_sg',
 '201202a_ndvi_sg',
 '201202b_ndvi_sg',
 '201203a_ndvi_sg',
 '201203b_ndvi_sg',
 '201204a_ndvi_sg',
 '201204b_ndvi_sg',
 '201205a_ndvi_sg',
 '201205b_ndvi_sg',
 '201206a_ndvi_sg',
 '201206b_ndvi_sg',
 '201207a_ndvi_sg',
 '201207b_ndvi_sg',
 '201208a_ndvi_sg',
 '201208b_ndvi_sg',
 '201209a_ndvi_sg',
 '201209b_ndvi_sg',
 '201210a_ndvi_sg',
 '201210b_ndvi_sg',
 '201211a_ndvi_sg',
 '201211b_ndvi_sg',
 '201212a_ndvi_sg',
 '201212b_ndvi_sg']

In [62]:
a = ee.Array(ps.first().get('array')).toList()

In [65]:
from scipy.signal import savgol_filter

In [93]:
sg_ps = ps.map(fun)

In [103]:
img_sg = sg_ps.reduceToImage(properties=[bands_sg.get(-1)], 
                             reducer=ee.Reducer.mean())\
        .reproject(images.first().projection().crs(),
                  images.first().projection().getInfo()['transform'])
Map.addLayer(img_sg, visParam, 'sg')

In [90]:
bands.get(0).cat('_sg').getInfo()

AttributeError: 'ComputedObject' object has no attribute 'cat'

In [95]:
sg_ps.first().getInfo()

{'type': 'Feature',
 'geometry': {'geodesic': False,
  'type': 'Point',
  'coordinates': [112.875, 29.208333333333336]},
 'id': '0',
 'properties': {'201201a_ndvi': 0.35499998927116394,
  '201201a_ndvi_sg': 0.28033333599984234,
  '201201b_ndvi': 0.3440000116825104,
  '201201b_ndvi_sg': 0.262902098241108,
  '201202a_ndvi': 0.24699999392032623,
  '201202a_ndvi_sg': 0.2976107252088739,
  '201202b_ndvi': 0.19200000166893005,
  '201202b_ndvi_sg': 0.35102331506344553,
  '201203a_ndvi': 0.30300000309944153,
  '201203a_ndvi_sg': 0.4331841537238279,
  '201203b_ndvi': 0.5450000166893005,
  '201203b_ndvi_sg': 0.512937069911779,
  '201204a_ndvi': 0.703000009059906,
  '201204a_ndvi_sg': 0.5433846224438061,
  '201204b_ndvi': 0.7080000042915344,
  '201204b_ndvi_sg': 0.5334056018393635,
  '201205a_ndvi': 0.5879999995231628,
  '201205a_ndvi_sg': 0.5087272782912065,
  '201205b_ndvi': 0.3050000071525574,
  '201205b_ndvi_sg': 0.4630769257778889,
  '201206a_ndvi': 0.13500000536441803,
  '201206a_ndvi_sg': 